# Reanalysis of NSCLC spatial-zone radiomics

This notebook recalculates feature selection, final Cox models, and C-indices for the max5, max10, and max20 settings using previously saved spatial-zone radiomics CSV files.

## 1. Upload the reanalysis package

Select `colab_reanalysis_en.zip`.

In [5]:
from google.colab import files
from pathlib import Path

In [1]:
! wget https://filedn.com/lpAczQGgeBjkX6l7SpI5JJy/public_code/nsclc_spatial_zone_radiomics/colab_reanalysis.zip

--2026-09-21 14:09:09--  https://filedn.com/lpAczQGgeBjkX6l7SpI5JJy/public_code/nsclc_spatial_zone_radiomics/colab_reanalysis.zip
Resolving filedn.com (filedn.com)... 74.120.8.113
Connecting to filedn.com (filedn.com)|74.120.8.113|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 41646124 (40M) [application/zip]
Saving to: ‘colab_reanalysis.zip’

colab_reanalysis.zi 100%[===================>]  39.72M  75.7MB/s    in 0.5s    

2026-09-21 14:09:10 (75.7 MB/s) - ‘colab_reanalysis.zip’ saved [41646124/41646124]



In [2]:
! unzip -q colab_reanalysis.zip

In [3]:
! ls

colab_reanalysis  colab_reanalysis.zip	sample_data


In [6]:
ROOT = Path('/content/colab_reanalysis/')
assert ROOT.is_dir(), ROOT

%cd /content/colab_reanalysis/

/content/colab_reanalysis


## 2. Install dependencies

In [7]:
%pip install -q -r requirements-colab.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 2.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 1.5 MB/s eta 0:00:00


## 3. Verify the input files

In [8]:
required = [
    ROOT / 'data/clinical_outcome/Supplementary-Table-S1-Training-set-LUNG1-Radiomics.csv',
    ROOT / 'data/clinical_outcome/Supplementary-Table-S2-Testing-set-LUNG1-Radiomics.csv',
    ROOT / 'data/clinical_outcome/Supplementary-Table-S3-Validation-set-LUNG2-Radiogenomics.csv',
    ROOT / 'data/radiomics_851_lung2/lung2_pyradiomics_851.csv',
    ROOT / 'data/common_cohort_manifest/aligned_case_manifest.csv',
    ROOT / 'data/spatial_features/features_physical2mm.csv',
    ROOT / 'data/spatial_features/features_physical3mm.csv',
    ROOT / 'data/spatial_features/features_physical5mm.csv',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))
print(f'Input-file check completed: {len(required)} files')

Input-file check completed: 8 files


## 4. Recalculate max5, max10, and max20

The same five-fold cross-validation is run for each maximum feature count, so this step may take several minutes.

In [9]:
import os
import subprocess
import sys

script = ROOT / 'reanalyze_strict_common_portable.py'
for cap in (5, 10, 20):
    output = ROOT / 'results' / f'max{cap}'
    output.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env['MAX_FEATURES'] = str(cap)
    env['STRICT_OUT'] = str(output)
    env['REANALYSIS_DATA'] = str(ROOT / 'data')
    print(f'Running max{cap} ...', flush=True)
    subprocess.run([sys.executable, str(script)], env=env, check=True)
print('All analyses have completed.')

Running max5 ...
Running max10 ...
Running max20 ...
All analyses have completed.


## 5. Compare the recalculated and original results

In [10]:
import numpy as np
import pandas as pd

def compare_cap(cap):
    generated = ROOT / 'results' / f'max{cap}'
    reference = ROOT / 'reference_results' / f'max{cap}'

    p_new = pd.read_csv(generated / 'performance.csv', encoding='utf-8-sig').sort_values(['condition', 'model', 'cohort']).reset_index(drop=True)
    p_ref = pd.read_csv(reference / 'performance.csv', encoding='utf-8-sig').sort_values(['condition', 'model', 'cohort']).reset_index(drop=True)
    performance_structure = p_new.drop(columns='c_index').equals(p_ref.drop(columns='c_index'))
    performance_numeric = np.allclose(p_new['c_index'], p_ref['c_index'], rtol=1e-10, atol=1e-12)

    s_new = pd.read_csv(generated / 'selected_features.csv', encoding='utf-8-sig')
    s_ref = pd.read_csv(reference / 'selected_features.csv', encoding='utf-8-sig')
    selected_equal = s_new.equals(s_ref)

    t_new = pd.read_csv(generated / 'elastic_net_tuning_summary.csv', encoding='utf-8-sig')
    t_ref = pd.read_csv(reference / 'elastic_net_tuning_summary.csv', encoding='utf-8-sig')
    tuning_structure = t_new.drop(columns='alpha').equals(t_ref.drop(columns='alpha'))
    tuning_numeric = np.allclose(t_new['alpha'], t_ref['alpha'], rtol=1e-12, atol=1e-14)

    return {
        'max_features': cap,
        'performance_equal': performance_structure and performance_numeric,
        'selected_features_equal': selected_equal,
        'tuning_equal': tuning_structure and tuning_numeric,
        'max_cindex_abs_diff': float(np.max(np.abs(p_new['c_index'] - p_ref['c_index']))),
    }

verification = pd.DataFrame([compare_cap(cap) for cap in (5, 10, 20)])
display(verification)
if not verification[['performance_equal', 'selected_features_equal', 'tuning_equal']].all().all():
    raise AssertionError('The recalculated results differ from the originals. Check the package versions and output files.')
print('The principal results match the original analysis.')

,max_features,performance_equal,selected_features_equal,tuning_equal,max_cindex_abs_diff
0,5,True,True,True,0.0
1,10,True,True,True,0.0
2,20,True,True,True,0.0


The principal results match the original analysis.


## 6. Display the external-validation results

In [11]:
tables = []
for cap in (5, 10, 20):
    table = pd.read_csv(ROOT / 'results' / f'max{cap}' / 'performance.csv', encoding='utf-8-sig')
    table = table.loc[table['cohort'].eq('LUNG2_validation'), ['condition', 'model', 'n', 'events', 'c_index']].copy()
    table.insert(0, 'max_features', cap)
    tables.append(table)
display(pd.concat(tables, ignore_index=True))

,max_features,condition,model,n,events,c_index
0,5,original_radiomics,radiomics,113,44,0.569956
1,5,original_radiomics,age_sex,113,44,0.587040
2,5,inner_ring_physical2mm,radiomics,113,44,0.604713
3,5,inner_ring_physical2mm,age_sex,113,44,0.630339
4,5,inner_core_physical2mm,radiomics,113,44,0.577025
5,5,inner_core_physical2mm,age_sex,113,44,0.602356
6,5,outer_ring_physical2mm,radiomics,113,44,0.594698
7,5,outer_ring_physical2mm,age_sex,113,44,0.617084
8,5,inner_ring_physical3mm,radiomics,113,44,0.597054
9,5,inner_ring_physical3mm,age_sex,113,44,0.614728


## 7. Download the recalculated results

Optional

In [ ]:
#import shutil
#archive = shutil.make_archive('/content/reanalysis_results', 'zip', ROOT / 'results')
#files.download(archive)